# Database Table Development

---

## I. Import libraries

In [ ]:
from cassandra.auth import PlainTextAuthProvider
from cassandra.query import SimpleStatement
from datetime import datetime, timezone
from cassandra import ConsistencyLevel
from cassandra.cluster import Cluster
from urllib.request import urlopen
from scipy.stats import zscore
from dotenv import load_dotenv
import yfinance as yf
import polars as pl
import pandas as pd
import numpy as np
import cassandra
import requests
import certifi
import time
import json
import csv
import re
import os

In [ ]:
load_dotenv("./.env", override=True)
URL_AVAILABLE_STOCK_SYMBOLS_WIKI = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
API_KEY_FMP = os.getenv("API_KEY_FMP")
DATABASE_WORK_PATH = "./Data/"

FMP_URL_COMPANY_INFOR = "https://financialmodelingprep.com/api/v3/search?query="
FMP_URL_STOCK_KEY_METRICS = "https://financialmodelingprep.com/api/v3/key-metrics/"
FMP_URL_MARKET_INDEX_LIST = "https://financialmodelingprep.com/stable/index-list?apikey="
FMP_URL_STOCK_SYMBOLS_CHANGE = "https://financialmodelingprep.com/api/v4/symbol_change?apikey="
FMP_URL_AVAILABLE_STOCK_SYMBOLS = "https://financialmodelingprep.com/api/v3/stock/list?apikey="
FMP_URL_MARKET_INDEX_QUOTE_DETAILS = "https://financialmodelingprep.com/api/v3/quotes/index?apikey="
FMP_URL_STOCK_COMPANY_PROFILE_DETAILS = "https://financialmodelingprep.com/api/v3/profile/"
FMP_URL_AVAILABLE_STOCK_SYMBOLS_SP500 = "https://financialmodelingprep.com/stable/sp500-constituent?apikey="
FMP_URL_AVAILABLE_STOCK_SYMBOLS_NASDAQ = "https://financialmodelingprep.com/stable/nasdaq-constituent?apikey="
FMP_URL_AVAILABLE_STOCK_SYMBOLS_DOWJONES = "https://financialmodelingprep.com/stable/dowjones-constituent?apikey="

## II. Create Cassandra Database

Create directory ./database
Inside directory ./database create node_1 node_2 directory 
docker pull cassandra
docker run --name cas1 -p 9042:9042 -v ./node_1:/var/lib/cassandra/data -e CASSANDRA_CLUSTER_NAME=WriteDatabase -e CASSANDRA_ENDPOINT_SNITCH=GossipingPropertyFileSnitch -e CASSANDRA_DC=datacenter_1 -d cassandra:latest
docker exec -it cas1 nodetool status
docker run --name cas2 -v ./node_2:/var/lib/cassandra/data -e CASSANDRA_SEEDS="$(docker inspect --format='{{ .NetworkSettings.IPAddress }}' cas1)" -e CASSANDRA_CLUSTER_NAME=WriteDatabase -e CASSANDRA_ENDPOINT_SNITCH=GossipingPropertyFileSnitch -e CASSANDRA_DC=datacenter_1 -d cassandra:latest

Note: Snitches determine how Cassandra distribute replicas. This snitch is recommended for production. GossipingPropertyFileSnitch automatically updates all nodes using gossip protocol when adding new nodes and is recommended for production. Run the following command to learn about the status of the node.

## III. Develop Bronze Layer

---

### A. Create Bronze Namespace

In [3]:
from cassandra.cluster import Cluster

cluster = Cluster([
    ('127.0.0.1', 9042),
    ('127.0.0.1', 9043)
])
session = cluster.connect()
try:
    session.execute("""
    CREATE KEYSPACE IF NOT EXISTS Test
    WITH REPLICATION = 
    { 'class' : 'NetworkTopologyStrategy', 'datacenter_1' : 2 }
    """)
    print("Keyspace created successfully.")
except Exception as e:
    print("Error creating keyspace:", e)

try:
    session.execute("USE Test")
    print("Keyspace set successfully.")
except Exception as e:
    print("Error setting keyspace:", e)

Keyspace created successfully.
Keyspace set successfully.


In [3]:
session.execute("USE Test")

In [ ]:
cqlsh -e "
CREATE TABLE Test.performance_metrics (
    node_ip TEXT PRIMARY KEY,
    read_latency DOUBLE,
    write_latency DOUBLE,
    request_count INT
)
"

In [ ]:
cqlsh -e "
INSERT INTO Test.performance_metrics (node_ip, read_latency, write_latency, request_count) 
VALUES ('127.0.0.1', 5.2, 7.8, 120)
"

In [7]:
try:
    session.execute("""
    CREATE TABLE IF NOT EXISTS users (
        id UUID PRIMARY KEY,
        name TEXT,
        email TEXT,
        age INT
    )
    """)
    print("Table created successfully.")
except Exception as e:
    print("Error creating table:", e)

Table created successfully.


In [11]:
import uuid

try:
    session.execute("""
    INSERT INTO users (id, name, email, age) 
    VALUES (%s, %s, %s, %s)
    """, (uuid.uuid4(), 'Alice Johnson', 'alice@example.com', 28))
    
    session.execute("""
    INSERT INTO users (id, name, email, age) 
    VALUES (%s, %s, %s, %s)
    """, (uuid.uuid4(), 'Bob Smith', 'bob@example.com', 32))
    
    print("Sample data inserted successfully.")
except Exception as e:
    print("Error inserting data:", e)

# Sample Query to Test
print("Run the following query to retrieve the data:")
print("SELECT * FROM Test.users;")

Sample data inserted successfully.
Run the following query to retrieve the data:
SELECT * FROM Test.users;


In [12]:
query = SimpleStatement("SELECT * FROM test.users;", consistency_level=ConsistencyLevel.QUORUM)
rows = session.execute(query)
for row in rows:
    print(row)

Row(id=UUID('1282d645-6850-42e6-a0f1-fcb72b85f1c9'), age=28, email='alice@example.com', name='Alice Johnson')
Row(id=UUID('b0f2dd54-7a93-44d7-ade9-f93b372d1af3'), age=28, email='alice@example.com', name='Alice Johnson')
Row(id=UUID('d6ecc327-cb8d-46f7-8110-4ff586499954'), age=32, email='bob@example.com', name='Bob Smith')
Row(id=UUID('10bcce9f-5270-4ab1-a041-78edd22d65a8'), age=32, email='bob@example.com', name='Bob Smith')
Row(id=UUID('539cd4a8-1b18-4be6-905b-f87cd37c1a39'), age=32, email='bob@example.com', name='Bob Smith')
Row(id=UUID('fb7096ad-94ce-4c6f-a210-af1460ce95f9'), age=32, email='bob@example.com', name='Bob Smith')
Row(id=UUID('b3b5ed9d-4814-4d25-b835-a9dbaca0b257'), age=32, email='bob@example.com', name='Bob Smith')
Row(id=UUID('e83a92d4-47f6-4ccf-89c2-b4f55040e888'), age=28, email='alice@example.com', name='Alice Johnson')
Row(id=UUID('1d96028f-9b97-4daf-824f-3d6d53de2974'), age=28, email='alice@example.com', name='Alice Johnson')
Row(id=UUID('2c95e2ae-34b0-4d8a-952a-3b5

In [16]:
query = "SELECT table_name FROM system_schema.tables WHERE keyspace_name = 'Test'"
rows = session.execute(query)

print("Existing Tables:")
for row in rows:
    print(row.table_name)

Existing Tables:


In [13]:
query = "SELECT node_ip, read_latency FROM performance_metrics"
rows = session.execute(query)
df_test = pd.DataFrame(rows, columns=["Node IP", "Read Latency (ms)"])

InvalidRequest: Error from server: code=2200 [Invalid query] message="table performance_metrics does not exist"

---

### B. Retrieve Data

In [4]:
def get_available_stock_symbols_market_data_FMP(url_available_stock_market, api_key):
    url_available_stock_market_api = url_available_stock_market + api_key
    response = urlopen(url_available_stock_market_api, cafile=certifi.where())
    data = response.read().decode("utf-8")
    data_df = pd.DataFrame(json.loads(data))
    return data_df

In [5]:
def ingest_stock_data(symbols, start="2010-01-01", end=pd.Timestamp.now().strftime("%Y-%m-%d")):
    """
    Fetch raw historical stock price data from Yahoo Finance.
    Returns a dictionary where keys are stock symbols and values are raw Pandas DataFrames.
    """
    stock_data = {}
    for symbol in symbols:
        try:
            data_yf = yf.download(symbol, start=start, end=end)
            # Rate-limiting to avoid getting blocked
            time.sleep(2)  
            
            if data_yf.empty:
                print(f"⚠️ No data found for {symbol}")
                stock_data[symbol] = None
                continue
            stock_data[symbol] = data_yf
        except Exception as e:
            print(f"❌ Error fetching {symbol}: {e}")
            stock_data[symbol] = None
    return stock_data

In [6]:
def transform_stock_data(stock_data):
    """
    Convert raw stock data from Pandas to Polars DataFrame.
    Returns a dictionary with structured data.
    """
    transformed_data = {}
    for symbol, raw_data in stock_data.items():
        if raw_data is None:
            transformed_data[symbol] = None
            continue
        try:
            pl_df = pl.DataFrame(raw_data.reset_index())
            pl_df = pl_df.rename({
                f"('Date', '')": "Date",
                f"('Close', '{symbol}')": "Close_Prices",
                f"('High', '{symbol}')": "High_Prices",
                f"('Low', '{symbol}')": "Low_Prices",
                f"('Open', '{symbol}')": "Open_Prices",
                f"('Volume', '{symbol}')": "Volume"
            })
            pl_df = pl_df.with_columns(pl.lit(symbol).alias("Symbol"))
            transformed_data[symbol] = pl_df
        except Exception as e:
            print(f"❌ Error transforming data for {symbol}: {e}")
            transformed_data[symbol] = None
    return transformed_data

In [7]:
def store_stock_data(stock_data, market_index):
    """
    Save transformed stock data to CSV files.
    """
    for symbol, processed_df in stock_data.items():
        if processed_df is None:
            continue
        try:
            directory = f"{DATABASE_WORK_PATH}{market_index}"
            os.makedirs(directory, exist_ok=True)
            file_path = f"{directory}/{symbol}.csv"
            if hasattr(processed_df, "to_csv"):
                processed_df.to_csv(file_path, index=False)
            elif hasattr(processed_df, "write_csv"):
                processed_df.write_csv(file_path)
            else:
                raise TypeError("Unsupported DataFrame type")
            print(f"✅ Successfully stored {symbol} at {file_path}")
        except Exception as e:
            print(f"❌ Error storing data for {symbol}: {e}")

#### 1. Historical Stock Prices Fact Tables

In [8]:
def fetch_historical_prices_dataframe_yfinance(symbols, market_index, start="2010-01-01", end=pd.Timestamp.now().strftime("%Y-%m-%d")):
    """
    End-to-end process:
    1. Fetch raw stock data.
    2. Transform raw data into structured Polars DataFrames.
    3. Store structured data in CSV format.
    """
    # Stage 1: Data Ingestion
    stock_data = ingest_stock_data(symbols, start, end)
    
    # Stage 2: Data Transformation
    transformed_data = transform_stock_data(stock_data)
    print("Historical Stock Prices Fact Tables Sample Columns\n", transformed_data[symbols[0]].columns)
    print("Historical Stock Prices Fact Tables Sample Dtypes\n", transformed_data[symbols[0]].dtypes)
    
    # Stage 3: Data Storage
    store_stock_data(transformed_data, market_index)

    return transformed_data

##### A. Historical Stock Prices Fact Tables (S&P500)

In [9]:
SP500_index_df = get_available_stock_symbols_market_data_FMP(FMP_URL_AVAILABLE_STOCK_SYMBOLS_SP500, API_KEY_FMP)
historical_stock_prices_sp500_fact_table_dict = fetch_historical_prices_dataframe_yfinance(SP500_index_df["symbol"].to_list()[:1], "SP500")

C:\Users\NolanM\AppData\Local\Temp\ipykernel_32876\2634297407.py:3: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  response = urlopen(url_available_stock_market_api, cafile=certifi.where())


YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


Historical Stock Prices Fact Tables Sample Columns
 ['Date', 'Close_Prices', 'High_Prices', 'Low_Prices', 'Open_Prices', 'Volume', 'Symbol']
Historical Stock Prices Fact Tables Sample Dtypes
 [Datetime(time_unit='ns', time_zone=None), Float64, Float64, Float64, Float64, Int64, String]
✅ Successfully stored APO at ./Data/SP500/APO.csv


In [10]:
null_stocks = [key for key, value in historical_stock_prices_sp500_fact_table_dict.items() if value is None]
print(f"Total Stocks Symbol Data Retrieve: {len(historical_stock_prices_sp500_fact_table_dict)}")
print(f"Stocks with null DataFrame: {null_stocks}")
print(f"Total count: {len(null_stocks)}")
print(historical_stock_prices_sp500_fact_table_dict['APO'].columns)
print(historical_stock_prices_sp500_fact_table_dict['APO'].dtypes)

Total Stocks Symbol Data Retrieve: 1
Stocks with null DataFrame: []
Total count: 0
['Date', 'Close_Prices', 'High_Prices', 'Low_Prices', 'Open_Prices', 'Volume', 'Symbol']
[Datetime(time_unit='ns', time_zone=None), Float64, Float64, Float64, Float64, Int64, String]


##### B. Historical Stock Prices Fact Tables (NASDAQ)

In [11]:
NASDAQ_index_df = get_available_stock_symbols_market_data_FMP(FMP_URL_AVAILABLE_STOCK_SYMBOLS_NASDAQ, API_KEY_FMP)
historical_stock_prices_nasdaq_fact_table_dict = fetch_historical_prices_dataframe_yfinance(NASDAQ_index_df["symbol"].to_list()[:1], "Nasdaq")

C:\Users\NolanM\AppData\Local\Temp\ipykernel_32876\2634297407.py:3: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  response = urlopen(url_available_stock_market_api, cafile=certifi.where())
[*********************100%***********************]  1 of 1 completed


Historical Stock Prices Fact Tables Sample Columns
 ['Date', 'Close_Prices', 'High_Prices', 'Low_Prices', 'Open_Prices', 'Volume', 'Symbol']
Historical Stock Prices Fact Tables Sample Dtypes
 [Datetime(time_unit='ns', time_zone=None), Float64, Float64, Float64, Float64, Int64, String]
✅ Successfully stored ADBE at ./Data/Nasdaq/ADBE.csv


In [12]:
null_stocks = [key for key, value in historical_stock_prices_nasdaq_fact_table_dict.items() if value is None]
print(f"Total Stocks Symbol Data Retrieve: {len(historical_stock_prices_nasdaq_fact_table_dict)}")
print(f"Stocks with null DataFrame: {null_stocks}")
print(f"Total count: {len(null_stocks)}")
print(historical_stock_prices_nasdaq_fact_table_dict['ADBE'].columns)
print(historical_stock_prices_nasdaq_fact_table_dict['ADBE'].dtypes)

Total Stocks Symbol Data Retrieve: 1
Stocks with null DataFrame: []
Total count: 0
['Date', 'Close_Prices', 'High_Prices', 'Low_Prices', 'Open_Prices', 'Volume', 'Symbol']
[Datetime(time_unit='ns', time_zone=None), Float64, Float64, Float64, Float64, Int64, String]


##### C. Historical Stock Prices Fact Tables (DOWJONES)

In [13]:
DOWJONES_index_df = get_available_stock_symbols_market_data_FMP(FMP_URL_AVAILABLE_STOCK_SYMBOLS_DOWJONES, API_KEY_FMP)
historical_stock_prices_dowjones_fact_table_dict = fetch_historical_prices_dataframe_yfinance(DOWJONES_index_df["symbol"].to_list()[:1], "Dowjones")

C:\Users\NolanM\AppData\Local\Temp\ipykernel_32876\2634297407.py:3: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  response = urlopen(url_available_stock_market_api, cafile=certifi.where())
[*********************100%***********************]  1 of 1 completed


Historical Stock Prices Fact Tables Sample Columns
 ['Date', 'Close_Prices', 'High_Prices', 'Low_Prices', 'Open_Prices', 'Volume', 'Symbol']
Historical Stock Prices Fact Tables Sample Dtypes
 [Datetime(time_unit='ns', time_zone=None), Float64, Float64, Float64, Float64, Int64, String]
✅ Successfully stored NVDA at ./Data/Dowjones/NVDA.csv


In [14]:
null_stocks = [key for key, value in historical_stock_prices_dowjones_fact_table_dict.items() if value is None]
print(f"Total Stocks Symbol Data Retrieve: {len(historical_stock_prices_dowjones_fact_table_dict)}")
print(f"Stocks with null DataFrame: {null_stocks}")
print(f"Total count: {len(null_stocks)}")
print(historical_stock_prices_dowjones_fact_table_dict['NVDA'].columns)
print(historical_stock_prices_dowjones_fact_table_dict['NVDA'].dtypes)

Total Stocks Symbol Data Retrieve: 1
Stocks with null DataFrame: []
Total count: 0
['Date', 'Close_Prices', 'High_Prices', 'Low_Prices', 'Open_Prices', 'Volume', 'Symbol']
[Datetime(time_unit='ns', time_zone=None), Float64, Float64, Float64, Float64, Int64, String]


---

#### 2. Stock Symbol Change Dimensional Table

In [15]:
symbol_stock_change_index_df = get_available_stock_symbols_market_data_FMP(FMP_URL_STOCK_SYMBOLS_CHANGE, API_KEY_FMP)
symbol_stock_change_index_path = f"{DATABASE_WORK_PATH}/Dimensional_tables"
os.makedirs(symbol_stock_change_index_path, exist_ok=True)
symbol_stock_change_index_df.to_csv(f"{symbol_stock_change_index_path}/historical_symbol_stock_change.csv")

C:\Users\NolanM\AppData\Local\Temp\ipykernel_32876\2634297407.py:3: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  response = urlopen(url_available_stock_market_api, cafile=certifi.where())


In [16]:
print(symbol_stock_change_index_df.columns)
print(symbol_stock_change_index_df.dtypes)

Index(['date', 'name', 'oldSymbol', 'newSymbol'], dtype='object')
date         object
name         object
oldSymbol    object
newSymbol    object
dtype: object


---

#### 3. Index Market Dimensional Table

In [17]:
list_index_market_group_path = f"{DATABASE_WORK_PATH}Dimensional_tables"
os.makedirs(list_index_market_group_path, exist_ok=True)
list_index_market_df = get_available_stock_symbols_market_data_FMP(FMP_URL_MARKET_INDEX_LIST, API_KEY_FMP)
list_index_market_df.to_csv(f"{list_index_market_group_path}/list_index_market.csv")
list_index_market_quote_df = get_available_stock_symbols_market_data_FMP(FMP_URL_MARKET_INDEX_QUOTE_DETAILS, API_KEY_FMP)
list_index_market_quote_df.to_csv(f"{list_index_market_group_path}/list_index_market_quote.csv")

C:\Users\NolanM\AppData\Local\Temp\ipykernel_32876\2634297407.py:3: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  response = urlopen(url_available_stock_market_api, cafile=certifi.where())


In [18]:
print(list_index_market_df.columns)
print(list_index_market_df.dtypes)
print("======================================================")
print(list_index_market_quote_df.columns)
print(list_index_market_quote_df.dtypes)

Index(['symbol', 'name', 'exchange', 'currency'], dtype='object')
symbol      object
name        object
exchange    object
currency    object
dtype: object
Index(['symbol', 'name', 'price', 'changesPercentage', 'change', 'dayLow',
       'dayHigh', 'yearHigh', 'yearLow', 'marketCap', 'priceAvg50',
       'priceAvg200', 'exchange', 'volume', 'avgVolume', 'open',
       'previousClose', 'eps', 'pe', 'earningsAnnouncement',
       'sharesOutstanding', 'timestamp'],
      dtype='object')
symbol                   object
name                     object
price                   float64
changesPercentage       float64
change                  float64
dayLow                  float64
dayHigh                 float64
yearHigh                float64
yearLow                 float64
marketCap               float64
priceAvg50              float64
priceAvg200             float64
exchange                 object
volume                    int64
avgVolume               float64
open                    float64

---

#### 4. Historical Financial Statement Fact Table
Bronze_Layer_Historical_Financial_Statement_Fact_Table_Flow

In [19]:
def ingest_quarter_historical_financial_statement_fact_data(symbols, period, api_key, limit=10000):
    """
    Fetch raw historical stock price data from Yahoo Finance.
    Returns a dictionary where keys are stock symbols and values are raw Pandas DataFrames.
    """
    quarter_historical_financial_statement_fact_dict = {}
    for symbol in symbols:
        try:
            url_quarter_historical_financial_statement_fact = FMP_URL_STOCK_KEY_METRICS + f"{symbol}?period={period}&limit={limit}&apikey={api_key}"
            response = urlopen(url_quarter_historical_financial_statement_fact, cafile=certifi.where())
            data = response.read().decode("utf-8")
            data_df_pl = pl.DataFrame(json.loads(data))
            # Rate-limiting to avoid getting blocked
            time.sleep(3)
            if data_df_pl.is_empty():
                print(f"⚠️ No data found for {symbol}")
                quarter_historical_financial_statement_fact_dict[symbol] = None
                continue
            quarter_historical_financial_statement_fact_dict[symbol] = data_df_pl
        except Exception as e:
            print(f"❌ Error fetching {symbol}: {e}")
            quarter_historical_financial_statement_fact_dict[symbol] = None
    return quarter_historical_financial_statement_fact_dict

In [20]:
def transform_quarter_historical_financial_statement_fact_data(quarter_historical_financial_statement_fact_data):
    """
    Convert raw stock data from Pandas to Polars DataFrame.
    Returns a dictionary with structured data.
    """
    transformed_data = {}
    for symbol, raw_data in quarter_historical_financial_statement_fact_data.items():
        if raw_data is None:
            transformed_data[symbol] = None
            continue
        try:
            pl_df = raw_data.unique(keep="any", maintain_order=True)
            transformed_data[symbol] = pl_df
        except Exception as e:
            print(f"❌ Error transforming data for {symbol}: {e}")
            transformed_data[symbol] = None
    return transformed_data

In [21]:
def store_quarter_historical_financial_statement_fact_data(quarter_historical_financial_statement_fact_data_process):
    """
    Save transformed stock data to CSV files.
    """
    for symbol, processed_df in quarter_historical_financial_statement_fact_data_process.items():
        if processed_df is None:
            continue
        try:
            directory = f"{DATABASE_WORK_PATH}Dimensional_tables/Financial_Statement_Historical_Fact_Table"
            os.makedirs(directory, exist_ok=True)  # Ensure directory exists
            file_path = f"{directory}/{symbol}.csv"
            processed_df.write_csv(file_path)
            print(f"✅ Successfully stored {symbol} at {file_path}")
        except Exception as e:
            print(f"❌ Error storing data for {symbol}: {e}")

In [22]:
test_symbols = DOWJONES_index_df["symbol"].to_list()[:2]
quarter_historical_financial_statement_fact_dict_ = ingest_quarter_historical_financial_statement_fact_data(test_symbols, "quarter", API_KEY_FMP, 10000)
quarter_historical_financial_statement_fact_dict_processed_ = transform_quarter_historical_financial_statement_fact_data(quarter_historical_financial_statement_fact_dict_)
store_quarter_historical_financial_statement_fact_data(quarter_historical_financial_statement_fact_dict_processed_)

C:\Users\NolanM\AppData\Local\Temp\ipykernel_32876\1108721653.py:10: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  response = urlopen(url_quarter_historical_financial_statement_fact, cafile=certifi.where())


✅ Successfully stored NVDA at ./Data/Dimensional_tables/Financial_Statement_Historical_Fact_Table/NVDA.csv
✅ Successfully stored SHW at ./Data/Dimensional_tables/Financial_Statement_Historical_Fact_Table/SHW.csv


In [32]:
print(quarter_historical_financial_statement_fact_dict_processed_['NVDA'].schema)

Schema({'symbol': String, 'date': String, 'calendarYear': String, 'period': String, 'revenuePerShare': Float64, 'netIncomePerShare': Float64, 'operatingCashFlowPerShare': Float64, 'freeCashFlowPerShare': Float64, 'cashPerShare': Float64, 'bookValuePerShare': Float64, 'tangibleBookValuePerShare': Float64, 'shareholdersEquityPerShare': Float64, 'interestDebtPerShare': Float64, 'marketCap': Float64, 'enterpriseValue': Float64, 'peRatio': Float64, 'priceToSalesRatio': Float64, 'pocfratio': Float64, 'pfcfRatio': Float64, 'pbRatio': Float64, 'ptbRatio': Float64, 'evToSales': Float64, 'enterpriseValueOverEBITDA': Float64, 'evToOperatingCashFlow': Float64, 'evToFreeCashFlow': Float64, 'earningsYield': Float64, 'freeCashFlowYield': Float64, 'debtToEquity': Float64, 'debtToAssets': Float64, 'netDebtToEBITDA': Float64, 'currentRatio': Float64, 'interestCoverage': Float64, 'incomeQuality': Float64, 'dividendYield': Float64, 'payoutRatio': Float64, 'salesGeneralAndAdministrativeToRevenue': Float64,

---

#### 5. Company Information Profile Stock Dimensional Table
Dimensional_Stock_Company_Information_Profile_Table_Tasks

In [23]:
def ingest_company_information_profile_dimensional_data(symbols, api_key):
    """
    Fetch raw company information profile stock dimensional
    Returns a dictionary where keys are stock symbols and values are raw Pandas DataFrames.
    """
    company_information_profile_dimensional_dict = {}
    for symbol in symbols:
        try:
            url_company_information_profile_dimensional = FMP_URL_STOCK_COMPANY_PROFILE_DETAILS + f"{symbol}?apikey={api_key}"
            response = urlopen(url_company_information_profile_dimensional, cafile=certifi.where())
            data = response.read().decode("utf-8")
            data_df_pl = pl.DataFrame(json.loads(data))
            # Rate-limiting to avoid getting blocked
            time.sleep(3)
            if data_df_pl.is_empty():
                print(f"⚠️ No data found for {symbol}")
                company_information_profile_dimensional_dict[symbol] = None
                continue

            for col in data_df_pl.columns:
                if data_df_pl[col].dtype == pl.Float64:
                    data_df_pl = data_df_pl.with_columns(data_df_pl[col].cast(pl.Float64))
                elif data_df_pl[col].dtype == pl.Int64:
                    data_df_pl = data_df_pl.with_columns(data_df_pl[col].cast(pl.Float64))
            
            company_information_profile_dimensional_dict[symbol] = data_df_pl
        except Exception as e:
            print(f"❌ Error fetching {symbol}: {e}")
            company_information_profile_dimensional_dict[symbol] = None

    dataframes_final = [df for df in company_information_profile_dimensional_dict.values() if df is not None]
    if dataframes_final:
        return pl.concat(dataframes_final, how="vertical")
    else:
        return pl.DataFrame()

In [24]:
def transform_company_information_profile_dimensional_data(company_information_profile_dimensional_df_data_raw, image_column="image"):
    """
    Download images from URLs in a Polars DataFrame and save them with custom filenames.
    """
    save_path= DATABASE_WORK_PATH + "stock_company_images"
    os.makedirs(save_path, exist_ok=True)
    pl_df = company_information_profile_dimensional_df_data_raw.unique(keep="any", maintain_order=True)
    for idx, row in enumerate(pl_df.iter_rows(named=True)):
        image_url = row.get(image_column)
        if not image_url:
            print(f"⚠️ Skipping empty URL at row {idx}")
            continue
        try:
            response = requests.get(image_url, stream=True)
            response.raise_for_status()
            filename = os.path.basename(image_url)
            file_path = os.path.join(save_path, filename)
            with open(file_path, "wb") as img_file:
                for chunk in response.iter_content(1024):
                    img_file.write(chunk)
            print(f"✅ Saved: {file_path}")
            return pl_df
        except requests.exceptions.RequestException as e:
            print(f"❌ Failed to download {image_url}: {e}")
            return pl_df

In [25]:
def store_company_information_profile_dimensional_data(company_information_profile_dimensional_df_data_processed):
    """
    Save transformed stock data to CSV files.
    """
    if company_information_profile_dimensional_df_data_processed is None:
        return "Empty Dataframe"
    try:
        directory = f"{DATABASE_WORK_PATH}Dimensional_tables/"
        os.makedirs(directory, exist_ok=True)  # Ensure directory exists
        file_path = f"{directory}company_information_profile_dimensional.csv"
        company_information_profile_dimensional_df_data_processed.write_csv(file_path)
        print(f"✅ Successfully stored company_information_profile_dimensional.csv at {file_path}")
    except Exception as e:
        print(f"❌ Error storing data for company_information_profile_dimensional.csv: {e}")

In [26]:
test_symbols = DOWJONES_index_df["symbol"].to_list()[:2]
company_information_profile_df = ingest_company_information_profile_dimensional_data(test_symbols, API_KEY_FMP)
company_information_profile_dimensional_df_data_processed_ = transform_company_information_profile_dimensional_data(company_information_profile_df)
store_company_information_profile_dimensional_data(company_information_profile_dimensional_df_data_processed_)

C:\Users\NolanM\AppData\Local\Temp\ipykernel_32876\2362242683.py:10: DeprecationWarning: cafile, capath and cadefault are deprecated, use a custom context instead.
  response = urlopen(url_company_information_profile_dimensional, cafile=certifi.where())


✅ Saved: ./Data/stock_company_images\NVDA.png
✅ Successfully stored company_information_profile_dimensional.csv at ./Data/Dimensional_tables/company_information_profile_dimensional.csv


In [27]:
company_information_profile_df[0].schema

Schema([('symbol', String),
        ('price', Float64),
        ('beta', Float64),
        ('volAvg', Float64),
        ('mktCap', Float64),
        ('lastDiv', Float64),
        ('range', String),
        ('changes', Float64),
        ('companyName', String),
        ('currency', String),
        ('cik', String),
        ('isin', String),
        ('cusip', String),
        ('exchange', String),
        ('exchangeShortName', String),
        ('industry', String),
        ('website', String),
        ('description', String),
        ('ceo', String),
        ('sector', String),
        ('country', String),
        ('fullTimeEmployees', String),
        ('phone', String),
        ('address', String),
        ('city', String),
        ('state', String),
        ('zip', String),
        ('dcfDiff', Float64),
        ('dcf', Float64),
        ('image', String),
        ('ipoDate', String),
        ('defaultImage', Boolean),
        ('isEtf', Boolean),
        ('isActivelyTrading', Boolean),


---

### C. Create Tables in Bronze Namespace

#### 1. Historical Stock Prices Fact Tables

In [ ]:
# try:
#     session.set_keyspace('Bronze_Historical_Stock_Prices_Nasdaq')
# except Exception as e:
#     print(e)

In [13]:
query_create_table_query_1 = """
    CREATE TABLE IF NOT EXISTS historical_stock_prices_test (
        date DATE,
        symbol TEXT,
        close_price DOUBLE,
        high_price DOUBLE,
        low_price DOUBLE,
        open_price DOUBLE,
        volume BIGINT,
        PRIMARY KEY ((symbol, date))
    );
"""
try:
    session.execute(query_create_table_query_1)
except Exception as e:
    print(e)

file_path = './Data/Dowjones/NVDA.csv'

insert_query = """
    INSERT INTO historical_stock_prices_test (date, symbol, close_price, high_price, low_price, open_price, volume)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
"""
try:
    with open(file_path, encoding='utf-8') as f:
        csvreader = csv.reader(f)
        # Skip header row
        next(csvreader)
        for line in csvreader:
            date_str = line[0]
            symbol = line[6]
            close_price = float(line[1])  
            high_price = float(line[2])  
            low_price = float(line[3])   
            open_price = float(line[4])  
            volume = int(line[5])       

            # Convert date to TIMESTAMP format
            date_obj = datetime.strptime(date_str, '%Y-%m-%dT%H:%M:%S.%f000').date()
            # Insert data into Cassandra
            session.execute(insert_query, (date_obj, symbol, close_price, high_price, low_price, open_price, volume))
    print("Data successfully inserted into historical_stock_prices_NVDA.")

except Exception as e:
    print(f"Error inserting data: {e}")

Data successfully inserted into historical_stock_prices_NVDA.


#### 2. Stock Symbol Change Dimensional Table

In [17]:
# try:
#     session.set_keyspace('Bronze_Layer')
# except Exception as e:
#     print(f"Error setting keyspace: {e}")
query_create_table = """
    CREATE TABLE IF NOT EXISTS historical_symbol_stock_change (
        date DATE,
        name TEXT,
        old_symbol TEXT,
        new_symbol TEXT,
        PRIMARY KEY ((old_symbol, date))
    )
"""
try:
    session.execute(query_create_table)
    print("Table historical_symbol_stock_change created successfully.")
except Exception as e:
    print(f"Error creating table: {e}")

symbol_stock_change_index_path = f"{DATABASE_WORK_PATH}/Dimensional_tables"
file_path = f"{symbol_stock_change_index_path}/historical_symbol_stock_change.csv"
insert_query = """
    INSERT INTO historical_symbol_stock_change (date, name, old_symbol, new_symbol)
    VALUES (%s, %s, %s, %s)
"""
try:
    with open(file_path, encoding='utf-8') as f:
        csvreader = csv.reader(f)
        next(csvreader)
        for line in csvreader:
            date_str = line[1]
            name = line[2]
            old_symbol = line[3]
            new_symbol = line[4]
            date_obj = datetime.strptime(date_str, '%Y-%m-%d').date()
            session.execute(insert_query, (date_obj, name, old_symbol, new_symbol))
    print("Data successfully inserted into historical_symbol_stock_change.")
except Exception as e:
    print(f"Error inserting data: {e}")

Table historical_symbol_stock_change created successfully.
Data successfully inserted into historical_symbol_stock_change.


#### 3. Index Market Dimensional Table

In [30]:
# try:
#     session.set_keyspace('Bronze_Layer')
# except Exception as e:
#     print(f"Error setting keyspace: {e}")
query_create_table = """
    CREATE TABLE IF NOT EXISTS list_index_market_quote_test (
        symbol TEXT,
        name TEXT,
        price DOUBLE,
        changes_percentage DOUBLE,
        change DOUBLE,
        day_low DOUBLE,
        day_high DOUBLE,
        year_high DOUBLE,
        year_low DOUBLE,
        market_cap DOUBLE,
        price_avg_50 DOUBLE,
        price_avg_200 DOUBLE,
        exchange TEXT,
        volume TEXT,
        avg_volume DOUBLE,
        open DOUBLE,
        previous_close DOUBLE,
        eps TEXT,
        pe TEXT,
        earnings_announcement TEXT,
        shares_outstanding TEXT,
        date DATE,
        PRIMARY KEY (symbol, date)
    )
"""
try:
    session.execute(query_create_table)
    print("Table list_index_market_quote created successfully.")
except Exception as e:
    print(f"Error creating table: {e}")

list_index_market_group_path = f"{DATABASE_WORK_PATH}Dimensional_tables"
file_path = f"{list_index_market_group_path}/list_index_market_quote.csv"
insert_query = """
    INSERT INTO list_index_market_quote_test (
        symbol, name, price, changes_percentage, change, day_low, day_high, 
        year_high, year_low, market_cap, price_avg_50, price_avg_200, exchange, 
        volume, avg_volume, open, previous_close, eps, pe, earnings_announcement, 
        shares_outstanding, date
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

try:
    with open(file_path, encoding='utf-8') as f:
        csvreader = csv.reader(f)
        next(csvreader)  # Skip header row

        for line in csvreader:
            symbol = line[1]
            name = line[2]
            price = float(line[3]) if line[3] else None
            changes_percentage = float(line[4]) if line[4] else None
            change = float(line[5]) if line[5] else None
            day_low = float(line[6]) if line[6] else None
            day_high = float(line[7]) if line[7] else None
            year_high = float(line[8]) if line[8] else None
            year_low = float(line[9]) if line[9] else None
            market_cap = float(line[10]) if line[10] else None
            price_avg_50 = float(line[11]) if line[11] else None
            price_avg_200 = float(line[12]) if line[12] else None
            exchange = line[13]
            volume = line[14] if line[14].isdigit() else None
            avg_volume = float(line[15]) if line[15] else None
            open_price = float(line[16]) if line[16] else None
            previous_close = float(line[17]) if line[17] else None
            eps = line[18] if line[18] else None
            pe = line[19] if line[19] else None
            earnings_announcement = line[20] if line[20] else None
            shares_outstanding = line[21] if line[21] else None
            date = line[22] if line[22] else None

            if date:
                try:
                    date_obj = datetime.utcfromtimestamp(int(date)).date()
                except ValueError:
                    print(f"Invalid date format: {date}")
                    date_obj = None
            else:
                date_obj = None
                
            print(date_obj)
            # Insert data into Cassandra
            session.execute(insert_query, (
                symbol, name, price, changes_percentage, change, day_low, day_high, 
                year_high, year_low, market_cap, price_avg_50, price_avg_200, exchange, 
                volume, avg_volume, open_price, previous_close, eps, pe, earnings_announcement, 
                shares_outstanding, date_obj
            ))

    print("Data successfully inserted into list_index_market_quote.")

except Exception as e:
    print(f"Error inserting data: {e}")

Table list_index_market_quote created successfully.
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2022-02-11
2022-02-11
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-25
2025-02-25
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2017-09-29
2017-09-29
2025-02-25
2025-02-25
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-26
2025-02-25
2025-02-25
2025-02-26
2025-02-26
20

#### 4. Historical Financial Statement Fact Table

In [42]:
# try:
#     session.set_keyspace('Bronze_Historical_Financial_Statement')
# except Exception as e:
#     print(f"Error setting keyspace: {e}")
query_create_table = """
    CREATE TABLE IF NOT EXISTS historical_financial_statement_test_test (
        symbol TEXT,
        date DATE,
        calendar_year TEXT,
        period TEXT,
        revenue_per_share DOUBLE,
        net_income_per_share DOUBLE,
        operating_cash_flow_per_share DOUBLE,
        free_cash_flow_per_share DOUBLE,
        cash_per_share DOUBLE,
        book_value_per_share DOUBLE,
        tangible_book_value_per_share DOUBLE,
        shareholders_equity_per_share DOUBLE,
        interest_debt_per_share DOUBLE,
        market_cap DOUBLE,
        enterprise_value DOUBLE,
        pe_ratio DOUBLE,
        price_to_sales_ratio DOUBLE,
        pocf_ratio DOUBLE,
        pfcf_ratio DOUBLE,
        pb_ratio DOUBLE,
        ptb_ratio DOUBLE,
        ev_to_sales DOUBLE,
        ev_to_ebitda DOUBLE,
        ev_to_operating_cash_flow DOUBLE,
        ev_to_free_cash_flow DOUBLE,
        earnings_yield DOUBLE,
        free_cash_flow_yield DOUBLE,
        debt_to_equity DOUBLE,
        debt_to_assets DOUBLE,
        net_debt_to_ebitda DOUBLE,
        current_ratio DOUBLE,
        interest_coverage DOUBLE,
        income_quality DOUBLE,
        dividend_yield DOUBLE,
        payout_ratio DOUBLE,
        sga_to_revenue DOUBLE,
        rnd_to_revenue DOUBLE,
        intangibles_to_assets DOUBLE,
        capex_to_operating_cash_flow DOUBLE,
        capex_to_revenue DOUBLE,
        capex_to_depreciation DOUBLE,
        stock_based_compensation_to_revenue DOUBLE,
        graham_number DOUBLE,
        roic DOUBLE,
        return_on_tangible_assets DOUBLE,
        graham_net_net DOUBLE,
        working_capital DOUBLE,
        tangible_asset_value DOUBLE,
        net_current_asset_value DOUBLE,
        invested_capital DOUBLE,
        average_receivables DOUBLE,
        average_payables DOUBLE,
        average_inventory DOUBLE,
        days_sales_outstanding DOUBLE,
        days_payables_outstanding DOUBLE,
        days_of_inventory_on_hand DOUBLE,
        receivables_turnover DOUBLE,
        payables_turnover DOUBLE,
        inventory_turnover DOUBLE,
        roe DOUBLE,
        capex_per_share DOUBLE,
        PRIMARY KEY (symbol, date)
    )
"""
try:
    session.execute(query_create_table)
    print("Table historical_financial_statement_NVDA created successfully.")
except Exception as e:
    print(f"Error creating table: {e}")
file_path = "./Data/Dimensional_tables/Financial_Statement_Historical_Fact_Table/NVDA.csv"
insert_query = """
    INSERT INTO historical_financial_statement_test_test (
        symbol, date, calendar_year, period, revenue_per_share, net_income_per_share, operating_cash_flow_per_share,
        free_cash_flow_per_share, cash_per_share, book_value_per_share, tangible_book_value_per_share,
        shareholders_equity_per_share, interest_debt_per_share, market_cap, enterprise_value, pe_ratio, price_to_sales_ratio,
        pocf_ratio, pfcf_ratio, pb_ratio, ptb_ratio, ev_to_sales, ev_to_ebitda, ev_to_operating_cash_flow, ev_to_free_cash_flow,
        earnings_yield, free_cash_flow_yield, debt_to_equity, debt_to_assets, net_debt_to_ebitda, current_ratio, interest_coverage,
        income_quality, dividend_yield, payout_ratio, sga_to_revenue, rnd_to_revenue, intangibles_to_assets, capex_to_operating_cash_flow,
        capex_to_revenue, capex_to_depreciation, stock_based_compensation_to_revenue, graham_number, roic, return_on_tangible_assets,
        graham_net_net, working_capital, tangible_asset_value, net_current_asset_value, invested_capital, average_receivables, average_payables, 
        average_inventory, days_sales_outstanding, days_payables_outstanding, days_of_inventory_on_hand, receivables_turnover, payables_turnover,
        inventory_turnover, roe, capex_per_share
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s)
"""
try:
    with open(file_path, encoding='utf-8') as f:
        csvreader = csv.reader(f)
        next(csvreader)  # Skip header row

        for line in csvreader:
            symbol = line[0]
            date = datetime.strptime(line[1], '%Y-%m-%d').date()
            calendar_year = line[2]
            period = line[3]
            numeric_values = [float(val) if val.strip() else None for val in line[4:]]
            
            session.execute(insert_query, (
                symbol, date, calendar_year, period, *numeric_values
            ))

    print("Data successfully inserted into historical_financial_statement_NVDA.")
except Exception as e:
    print(f"Error inserting data: {e}")

Table historical_financial_statement_NVDA created successfully.
Data successfully inserted into historical_financial_statement_NVDA.


#### 5. Company Information Profile Stock Dimensional Table

In [52]:
# try:
#     session.set_keyspace('Bronze_Layer')
# except Exception as e:
#     print(f"Error setting keyspace: {e}")

# Define the table creation query
query_create_table = """
    CREATE TABLE IF NOT EXISTS company_information_profile (
        symbol TEXT PRIMARY KEY,
        price DOUBLE,
        beta DOUBLE,
        vol_avg DOUBLE,
        market_cap DOUBLE,
        last_div DOUBLE,
        range TEXT,
        changes DOUBLE,
        company_name TEXT,
        currency TEXT,
        cik TEXT,
        isin TEXT,
        cusip TEXT,
        exchange TEXT,
        exchange_short_name TEXT,
        industry TEXT,
        website TEXT,
        description TEXT,
        ceo TEXT,
        sector TEXT,
        country TEXT,
        full_time_employees TEXT,
        phone TEXT,
        address TEXT,
        city TEXT,
        state TEXT,
        zip TEXT,
        dcf_diff DOUBLE,
        dcf DOUBLE,
        image TEXT,
        ipo_date DATE,
        default_image BOOLEAN,
        is_etf BOOLEAN,
        is_actively_trading BOOLEAN,
        is_adr BOOLEAN,
        is_fund BOOLEAN
    )
"""
try:
    session.execute(query_create_table)
    print("Table company_information_profile created successfully.")
except Exception as e:
    print(f"Error creating table: {e}")

# File path
file_path = "./Data/Dimensional_tables/company_information_profile_dimensional.csv"

# Insert data into the table
insert_query = """
    INSERT INTO company_information_profile (
        symbol, price, beta, vol_avg, market_cap, last_div, range, changes, company_name,
        currency, cik, isin, cusip, exchange, exchange_short_name, industry, website,
        description, ceo, sector, country, full_time_employees, phone, address, city,
        state, zip, dcf_diff, dcf, image, ipo_date, default_image, is_etf,
        is_actively_trading, is_adr, is_fund
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

try:
    with open(file_path, encoding='utf-8') as f:
        csvreader = csv.reader(f)
        next(csvreader)  # Skip header row

        for line in csvreader:
            symbol = line[0]
            price = float(line[1].strip()) if line[1].strip() else None
            beta = float(line[2].strip()) if line[2].strip() else None
            vol_avg = float(line[3].strip()) if line[3].strip() else None
            market_cap = float(line[4].strip()) if line[4].strip() else None
            last_div = float(line[5].strip()) if line[5].strip() else None
            range_value = line[6]
            changes = float(line[7].strip()) if line[7].strip() else None
            company_name = line[8]
            currency = line[9]
            cik = line[10]
            isin = line[11]
            cusip = line[12]
            exchange = line[13]
            exchange_short_name = line[14]
            industry = line[15]
            website = line[16]
            description = line[17]
            ceo = line[18]
            sector = line[19]
            country = line[20]
            full_time_employees = line[21]
            phone = line[22]
            address = line[23]
            city = line[24]
            state = line[25]
            zip_code = line[26]
            dcf_diff = float(line[27].strip()) if line[27].strip() else None
            dcf = float(line[28].strip()) if line[28].strip() else None
            image = line[29] if line[29].strip() else None
            ipo_date = datetime.strptime(line[30].strip(), '%Y-%m-%d').date() if line[30].strip() else None
            default_image = line[31].strip().lower() == 'true' if line[31].strip() else False
            is_etf = line[32].strip().lower() == 'true' if line[32].strip() else False
            is_actively_trading = line[33].strip().lower() == 'true' if line[33].strip() else False
            is_adr = line[34].strip().lower() == 'true' if line[34].strip() else False
            is_fund = line[35].strip().lower() == 'true' if line[35].strip() else False

            session.execute(insert_query, (
                symbol, price, beta, vol_avg, market_cap, last_div, range_value, changes, company_name,
                currency, cik, isin, cusip, exchange, exchange_short_name, industry, website,
                description, ceo, sector, country, full_time_employees, phone, address, city,
                state, zip_code, dcf_diff, dcf, image, ipo_date, default_image, is_etf,
                is_actively_trading, is_adr, is_fund
            ))

    print("Data successfully inserted into company_information_profile.")

except Exception as e:
    print(f"Error inserting data: {e}")

Table company_information_profile created successfully.
Data successfully inserted into company_information_profile.
